# Reproducible Python projects: environments, packages, and versions

Open the machinery behind the command used in Lectures 1 and 2, then build an environment
provenance manifest for a scientific computation.

**Lecture 3 · Notebook 00 · CMOR 438 / INDE 577**

## Orientation

**Core:** the project model, declared and locked dependencies, virtual environments, kernels,
imports, and the provenance manifest.

**Practice:** diagnose an environment and compare installed versions with the lockfile.

**Extension:** decide what a research-grade manifest must capture across hardware, data, and
nondeterministic computation.

This notebook explains infrastructure rather than NumPy. Notebook 01 moves from scripts to modules
and packages; later notebooks use NumPy and pandas inside the environment established here.

## How to use this notebook

**Estimated time:** 60 minutes core, plus 35 minutes of practice and extension.

**Prerequisites:** Lectures 1–2 and the working **Rice DSM** kernel. No terminal expertise beyond
running the course setup command is assumed.

Predict each diagnostic before running it. Paths will differ across computers; relationships should
not. The notebook performs read-only inspection and writes derived output only to a temporary
directory. It never installs, upgrades, or removes a dependency.

Companion reading: [How the course Python project works](../../supplementary-materials/computing-foundations/06-how-the-course-python-project-works.md).

## Learning objectives

By the end, you should be able to:

- distinguish a repository, project, interpreter, environment, kernel, distribution, import
  package, module, and notebook;
- trace `uv run python scripts/setup_course.py` across process and environment boundaries;
- read project intent from `pyproject.toml` and a resolution from `uv.lock`;
- distinguish declared, locked, installed, imported, and running versions;
- verify which environment and source tree execute a notebook;
- explain editable installation without treating `sys.path` modification as a repair;
- diagnose location, synchronization, kernel, import, and stale-state failures; and
- create a validated, serializable provenance manifest for a computation.

## Why this matters in industry and science

“It worked on my laptop” is not evidence another researcher can reproduce a result. A prediction can
change because of code, input data, configuration, Python, a direct dependency, a transitive
dependency, an operating-system-specific build, hardware, or random state.

A notebook records code and prose, but not a complete live environment. Professional work makes the
execution context inspectable and reconstructable. A lockfile narrows one source of variation; it
does not establish scientific validity, data provenance, or numerical equivalence across hardware.

## Worked example: provenance for a computational result

Imagine two teams obtain different estimates from the same model notebook. Before debating the
mathematics, they need evidence about what actually ran. We will build this chain:

```text
repository files
    → project requirements
    → cross-platform lock resolution
    → machine-local .venv
    → selected Jupyter kernel
    → imported source
    → environment provenance manifest
```

The manifest will be an **observation of one run**, not a claim that the run is correct or fully
reproducible.

## Professional practice

| Data scientist asks | Software engineer asks |
| --- | --- |
| Which code and data produced this result? | Which revision and paths identify them? |
| Could a numerical library change the estimate? | Which resolved and installed versions ran? |
| Was randomness controlled? | Where is seed and configuration recorded? |
| Could preprocessing differ between teams? | Is the pipeline packaged behind one interface? |
| Does another platform give equivalent results? | Does CI exercise every supported platform? |
| Is the result scientifically valid? | Can the environment be reconstructed and diagnosed? |

Reproducible software is necessary for credible computational science, but it is not sufficient.

## 1. Name the layers precisely

| Layer | What it is in this course |
| --- | --- |
| repository | the Git-controlled course directory and history |
| Python project | metadata, requirements, build configuration, source, and tests |
| interpreter | one Python executable running instructions |
| virtual environment | a project-local installation context rooted at `.venv` |
| distribution package | installable project named `rice-dsm` |
| import package | importable namespace named `rice_dsm` |
| module | one importable unit, often a `.py` file |
| kernel | a long-running interpreter process connected to the notebook UI |
| notebook | a JSON document containing cells and metadata—not a process |

These layers cooperate, but none is a synonym for another.

### System map

```text
VS Code notebook editor
        │ sends a cell through the Jupyter extension
        ▼
Rice DSM kernelspec
        │ launches
        ▼
.venv Python kernel ──imports──> installed distributions + repository/src
        ▲
        │ environment synchronized from
pyproject.toml + uv.lock
```

The shell used during setup exits. The kernel later launched by VS Code is a different process.

In [ ]:
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import platform
import subprocess
import sys
import sysconfig
import tomllib
from collections.abc import Iterable, Mapping
from dataclasses import asdict, dataclass
from pathlib import Path
from tempfile import TemporaryDirectory

## 2. Establish the project boundary

Tools begin from a current working directory and search for configuration. Notebook kernels can
start in different directories, so robust code constructs paths from a verified project root rather
than a personal home directory.

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest Rice DSM project directory.

    Parameters
    ----------
    start : Path
        Directory from which to search upward.

    Returns
    -------
    Path
        Directory containing this course's ``pyproject.toml``.

    Raises
    ------
    FileNotFoundError
        If no matching project is found.
    """

    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        project_file = candidate / "pyproject.toml"
        if project_file.is_file() and "rice-dsm" in project_file.read_text(
            encoding="utf-8"
        ):
            return candidate
    raise FileNotFoundError(f"rice-dsm project not found above {resolved_start}")


working_directory = Path.cwd()
project_root = find_project_root(working_directory)
print("Kernel working directory:", working_directory)
print("Verified project root:    ", project_root)

In [ ]:
project_paths = {
    "project declaration": project_root / "pyproject.toml",
    "lock resolution": project_root / "uv.lock",
    "Python request": project_root / ".python-version",
    "local environment": project_root / ".venv",
    "import source": project_root / "src" / "rice_dsm",
    "behavioral tests": project_root / "tests",
}

for role, path in project_paths.items():
    status = "found" if path.exists() else "MISSING"
    print(f"{status:>7} | {role:<19} | {path.relative_to(project_root)}")

assert all(path.exists() for path in project_paths.values())

### Common failure: searching for the wrong project

Finding *any* `pyproject.toml` is weaker than finding this project. Parent directories or neighboring
projects may have their own configuration. Our helper checks a stable course identifier and reports
the searched location when it fails.

In [ ]:
with TemporaryDirectory() as empty_directory:
    try:
        find_project_root(Path(empty_directory))
    except FileNotFoundError as error:
        print(f"Expected diagnostic: {error}")
    else:
        message = "an unrelated temporary directory is not the course project"
        raise AssertionError(message)

## 3. `pyproject.toml` declares intent

TOML is a configuration format. Python 3.11+ includes the read-only `tomllib` parser. The major
tables have different owners:

- `[build-system]`: how source becomes a distribution;
- `[project]`: standardized project metadata and runtime requirements;
- `[dependency-groups]`: development environments such as testing and notebook tools; and
- `[tool.*]`: configuration interpreted by a named tool.

Valid TOML is only syntax. We still check the fields our workflow requires.

In [ ]:
pyproject_path = project_paths["project declaration"]
with pyproject_path.open(mode="rb") as handle:
    project_document = tomllib.load(handle)

print("Top-level tables:", sorted(project_document))
print("Project name:    ", project_document["project"]["name"])
print("Project version: ", project_document["project"]["version"])
print("Python policy:   ", project_document["project"]["requires-python"])
print("Build backend:   ", project_document["build-system"]["build-backend"])

assert project_document["project"]["name"] == "rice-dsm"
assert project_document["build-system"]["build-backend"] == "uv_build"

### Runtime dependencies and development dependencies answer different questions

`[project].dependencies` belongs to users of the `rice-dsm` distribution. The `dev` dependency group
supports people developing or teaching from this repository: kernel, notebook execution, tests, and
linting. NumPy, pandas, Matplotlib, Seaborn, and Plotly are runtime requirements because course
package code and analytical notebooks may rely on them. SQLAlchemy, DuckDB, and Psycopg support
the database lesson. Boto3, PyArrow, and Polars support cloud object-storage, columnar scanning,
and lazy larger-than-memory workflows. FastAPI, HTTPX2, and Uvicorn support the package-backed
end-to-end data-product lesson. The OpenAI client supports the hosted web API—a documented
request/response interface between programs—in the offline-first LLM agent lesson. Shipping every
developer tool as a runtime requirement would burden
downstream users.

In [ ]:
runtime_requirements = tuple(project_document["project"]["dependencies"])
development_requirements = tuple(project_document["dependency-groups"]["dev"])

print("Runtime requirements:", runtime_requirements)
print("Development requirements:")
for requirement in development_requirements:
    print(" -", requirement)

runtime_names = {
    requirement.split(">=", maxsplit=1)[0]
    for requirement in runtime_requirements
}
assert {
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "plotly",
    "sqlalchemy",
    "duckdb",
    "psycopg[binary]",
    "boto3",
    "pyarrow",
    "polars",
    "fastapi",
    "httpx2",
    "uvicorn",
    "openai",
} <= runtime_names
assert any(requirement.startswith("pytest") for requirement in development_requirements)

## 4. `uv.lock` records a resolution

Requirements describe acceptable versions; a resolver chooses a compatible dependency graph. The
cross-platform `uv.lock` stores exact package versions, sources, hashes, dependency edges, and
environment markers needed across supported platforms.

The lockfile is committed and managed by `uv`. Read it for diagnosis; do not hand-edit it. New
releases do not automatically make an existing lockfile stale.

In [ ]:
lock_path = project_paths["lock resolution"]
with lock_path.open(mode="rb") as handle:
    lock_document = tomllib.load(handle)

locked_packages = lock_document["package"]
locked_versions = {
    package["name"]: package.get("version")
    for package in locked_packages
}

print("Lock format version:", lock_document["version"])
print("Python policy:      ", lock_document["requires-python"])
print("Package records:    ", len(locked_packages))
print("Locked pytest:      ", locked_versions["pytest"])

assert (
    lock_document["requires-python"]
    == project_document["project"]["requires-python"]
)
assert locked_versions["rice-dsm"] == project_document["project"]["version"]

### Declared, locked, installed, imported, and running

These states are related but distinct:

| State | Evidence |
| --- | --- |
| declared acceptable range | `pyproject.toml` requirement |
| resolved exact version | `uv.lock` package record |
| installed distribution | metadata inside this environment |
| imported module | object and `__file__` in this process |
| running Python | `sys.version` and `sys.prefix` |

A debugging claim should name its evidence. “I have version 3.12” is ambiguous until we know what is
being versioned and which process reported it.

In [ ]:
@dataclass(frozen=True, slots=True)
class VersionObservation:
    """A locked version compared with one installed environment.

    Parameters
    ----------
    distribution_name : str
        Normalized distribution name.
    locked_version : str or None
        Version selected in the lockfile.
    installed_version : str or None
        Version reported by the active environment.
    """

    distribution_name: str
    locked_version: str | None
    installed_version: str | None

    @property
    def agrees(self) -> bool:
        """Return whether both versions exist and match exactly."""

        return (
            self.locked_version is not None
            and self.locked_version == self.installed_version
        )


def observe_versions(
    distribution_names: Iterable[str],
    lock_versions: Mapping[str, object],
) -> tuple[VersionObservation, ...]:
    """Compare selected lockfile records with installed metadata.

    Parameters
    ----------
    distribution_names : iterable of str
        Distribution names to inspect.
    lock_versions : mapping of str to object
        Exact versions parsed from a lockfile.

    Returns
    -------
    tuple of VersionObservation
        One immutable comparison per requested name.
    """

    observations: list[VersionObservation] = []
    for name in distribution_names:
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = None
        locked = lock_versions.get(name)
        observations.append(VersionObservation(
            distribution_name=name,
            locked_version=locked if isinstance(locked, str) else None,
            installed_version=installed,
        ))
    return tuple(observations)

In [ ]:
version_observations = observe_versions(
    ("rice-dsm", "pytest", "ruff", "nbformat"),
    locked_versions,
)
for observation in version_observations:
    symbol = "✓" if observation.agrees else "!"
    print(
        f"{symbol} {observation.distribution_name:<10} "
        f"locked={observation.locked_version!s:<8} "
        f"installed={observation.installed_version}"
    )

assert all(observation.agrees for observation in version_observations)

### A universal lockfile is not a list of packages all machines install

`uv.lock` can contain platform markers and wheels for Windows, macOS, Linux, multiple architectures,
or Python versions. One `.venv` installs the subset selected for its platform and enabled dependency
groups. Therefore, “every locked package must be installed here” is a false invariant.

Likewise, equal versions do not guarantee bitwise-identical floating-point output across all
hardware. Versions are essential provenance, not the whole reproducibility story.

## 5. Locking, syncing, and running are different operations

```text
pyproject.toml ──resolve/lock──> uv.lock ──sync──> .venv ──run──> process
       ▲
       └──────────── uv add changes project intent
```

- `uv add PACKAGE` changes the shared project requirements, lockfile, and environment.
- `uv lock` resolves requirements into the lockfile.
- `uv sync --locked` refuses to update a stale lockfile and synchronizes `.venv`.
- `uv run COMMAND` checks project state, synchronizes as needed, then runs the command.

In CI, `--locked` turns unexpected project/lock disagreement into a failure instead of a mutation.

### An important edge case: exact versus inexact synchronization

Current `uv` behavior distinguishes two defaults:

- `uv sync` is **exact** by default and removes extraneous packages;
- `uv run` is **inexact** by default and ensures required packages exist without necessarily removing
  extras.

This is one reason not to modify `.venv` manually. An undeclared package might appear to work on one
laptop and disappear during exact synchronization or CI. If code imports it, declare it properly.

### Common failure: confusing `uv add` with `uv sync`

When an instructor commits a new dependency, students **sync** the existing project definition.
They do not each **add** it again. Adding is a design change that can rewrite shared project files;
syncing reconstructs what those files already specify.

Never install a package merely by guessing from an import name. Distribution and import names need
not match, and installing an unintended name is a supply-chain risk.

## 6. Inspect the interpreter and virtual environment

`sys.executable` identifies the interpreter program. `sys.prefix` identifies the active installation
context. `sys.base_prefix` identifies the base Python from which a virtual environment was created.

With uv-managed Python, the executable can resolve to a shared uv installation while `sys.prefix`
correctly points to the project `.venv`. Test the environment relationship—not whether an arbitrary
path string contains `.venv`.

In [ ]:
expected_environment = (project_root / ".venv").resolve()
active_environment = Path(sys.prefix).resolve()
interpreter = Path(sys.executable)
using_virtual_environment = sys.prefix != sys.base_prefix
using_course_environment = active_environment == expected_environment

print("Python version:       ", platform.python_version())
print("Interpreter:          ", interpreter)
print("Resolved interpreter: ", interpreter.resolve())
print("Base prefix:          ", sys.base_prefix)
print("Active prefix:        ", sys.prefix)
print("Expected course .venv:", expected_environment)
print("Virtual environment:  ", using_virtual_environment)
print("Course environment:   ", using_course_environment)

assert sys.version_info[:2] == (3, 12)
assert using_virtual_environment
assert using_course_environment

In [ ]:
purelib_directory = Path(sysconfig.get_path("purelib")).resolve()
print("Third-party distributions live under:")
print(purelib_directory)

assert purelib_directory.is_relative_to(expected_environment)

### Activation is shell convenience, not environment creation

Activation changes the current shell's command search path so bare `python` finds `.venv`. It does
not create the environment or install dependencies. Activation commands differ across shells.

This course uses `uv run ...` for one cross-platform command path. VS Code starts the selected
kernel directly, so the integrated terminal does not need to remain activated.

## 7. VS Code, Jupyter, kernelspec, and kernel

VS Code displays and edits the `.ipynb` file. Its Jupyter extension reads a **kernelspec**, which is
a small launch description. Selecting **Rice DSM** tells the extension which Python process to
start. The running **kernel** executes cells and owns their live objects.

Installing a dependency in one environment cannot change a kernel already running another
interpreter. A kernelspec name is helpful evidence, but `sys.prefix` is the decisive runtime check.

In [ ]:
from jupyter_client.kernelspec import KernelSpecManager

kernel_specification = KernelSpecManager().get_kernel_spec("rice-dsm")
kernel_command = kernel_specification.argv
kernel_interpreter = Path(kernel_command[0]).absolute()

print("Display name:", kernel_specification.display_name)
print("Launch command:", kernel_command)

assert kernel_specification.display_name == "Rice DSM"
assert kernel_interpreter.is_relative_to(project_root / ".venv")

### Kernel memory explains out-of-order notebook bugs

The kernel is long-running. Names created by an earlier cell remain until deletion or restart.
Notebook output saved on disk is only a representation; it is not live memory. “Restart and Run
All” tests whether the document contains a reproducible execution order.

In [ ]:
experiment_configuration = {"seed": 438, "threshold": 0.8}
configuration_identity = id(experiment_configuration)
print("Stored configuration object:", configuration_identity)

In [ ]:
print("Later cell sees the same object:", id(experiment_configuration))
assert id(experiment_configuration) == configuration_identity

If you restart and execute only the second cell, it fails with `NameError`. That is not an environment
failure: the correct kernel simply has fresh memory. Diagnose process state separately from package
installation.

## 8. Distribution name, import package, and module origin

Packaging tools install the `rice-dsm` **distribution**. Python imports the `rice_dsm` **package**.
The relationship is metadata, not a universal spelling rule: one distribution can provide multiple
imports, and names can differ entirely.

`importlib.metadata` inspects distributions. `importlib.util.find_spec` asks how the import system
would locate a module without importing it. The convenience mapping from import names to
distributions can be incomplete for editable installs, so absence from that mapping is not proof
that the distribution or import is missing.

In [ ]:
distribution_name = "rice-dsm"
import_name = "rice_dsm"

course_distribution = importlib.metadata.distribution(distribution_name)
course_specification = importlib.util.find_spec(import_name)
provided_by = importlib.metadata.packages_distributions().get(import_name, [])

assert course_specification is not None
print("Distribution: ", course_distribution.metadata["Name"])
print("Version:      ", course_distribution.version)
print("Import name:  ", import_name)
print("Provided by:  ", provided_by)
print("Import origin:", course_specification.origin)

if not provided_by:
    print("Note: editable installs may omit top-level-name mapping metadata.")

assert course_distribution.metadata["Name"] == "rice-dsm"

In [ ]:
rice_dsm = importlib.import_module(import_name)
import_origin = Path(rice_dsm.__file__).resolve()
expected_source = (project_root / "src" / "rice_dsm").resolve()

print("Imported version:", rice_dsm.__version__)
print("Imported source: ", import_origin)

assert import_origin.is_relative_to(expected_source)
assert rice_dsm.__version__ == course_distribution.version

### Editable installation connects installed metadata to working source

During sync, the build backend installs this project in editable form. Import resolution points to
`src/rice_dsm/`, so source edits need not be recopied into `.venv`. A running process can still cache
an already imported module; restart the kernel after source edits when behavior appears stale.

Do not repair imports by inserting personal paths into `sys.path` inside a notebook. That hides a
broken environment and creates a document that works only on one filesystem.

In [ ]:
direct_url_text = course_distribution.read_text("direct_url.json")
assert direct_url_text is not None
direct_url = json.loads(direct_url_text)

print("Editable install:", direct_url["dir_info"]["editable"])
print("Source URL:      ", direct_url["url"])

assert direct_url["dir_info"]["editable"] is True

### Verify imports in a fresh process outside the repository

Importing successfully from the repository root can be misleading because the working directory may
help Python find local files. A fresh subprocess with a temporary working directory tests the
installed connection. Passing a list of arguments avoids platform-specific shell quoting.

In [ ]:
with TemporaryDirectory() as outside_repository:
    completed = subprocess.run(
        [
            sys.executable,
            "-c",
            "import rice_dsm; print(rice_dsm.__file__)",
        ],
        cwd=outside_repository,
        check=True,
        capture_output=True,
        text=True,
    )

fresh_process_origin = Path(completed.stdout.strip()).resolve()
print("Fresh process imported:", fresh_process_origin)
assert fresh_process_origin.is_relative_to(expected_source)

## 9. Worked example: build an environment provenance manifest

We now combine the evidence into an immutable record. Its invariants catch missing or malformed
identity information. A SHA-256 digest fingerprints the exact lockfile bytes; it is an identifier
for comparison, not proof that the dependencies are safe.

In [ ]:
def sha256_file(path: Path) -> str:
    """Compute the SHA-256 hexadecimal digest of a file.

    Parameters
    ----------
    path : Path
        File to read as bytes.

    Returns
    -------
    str
        Sixty-four-character lowercase hexadecimal digest.

    Raises
    ------
    OSError
        If the file cannot be read.
    """

    digest = hashlib.sha256()
    with path.open(mode="rb") as handle:
        for chunk in iter(lambda: handle.read(65_536), b""):
            digest.update(chunk)
    return digest.hexdigest()


@dataclass(frozen=True, slots=True)
class EnvironmentManifest:
    """Provenance describing the software context of one computation.

    Parameters
    ----------
    project_distribution : str
        Installed distribution name.
    project_version : str
        Installed distribution version.
    python_version : str
        Running Python version.
    platform_description : str
        Operating system and architecture description.
    environment_prefix : str
        Active Python installation context.
    import_package : str
        Public import name.
    import_origin : str
        Source file loaded by the running process.
    lock_sha256 : str
        Digest of the complete lockfile bytes.
    direct_requirements : tuple of str
        Requirements declared for runtime use.

    Raises
    ------
    TypeError
        If a field has the wrong runtime type.
    ValueError
        If required identity evidence is blank or malformed.
    """

    project_distribution: str
    project_version: str
    python_version: str
    platform_description: str
    environment_prefix: str
    import_package: str
    import_origin: str
    lock_sha256: str
    direct_requirements: tuple[str, ...]

    def __post_init__(self) -> None:
        """Validate the completed manifest."""

        text_values = (
            self.project_distribution,
            self.project_version,
            self.python_version,
            self.platform_description,
            self.environment_prefix,
            self.import_package,
            self.import_origin,
        )
        if not all(isinstance(value, str) and value.strip() for value in text_values):
            raise TypeError("manifest identity fields must be nonblank strings")
        if (
            not isinstance(self.lock_sha256, str)
            or len(self.lock_sha256) != 64
            or any(
                character not in "0123456789abcdef"
                for character in self.lock_sha256
            )
        ):
            raise ValueError("lock_sha256 must be a lowercase SHA-256 digest")
        if not isinstance(self.direct_requirements, tuple) or not all(
            isinstance(requirement, str) for requirement in self.direct_requirements
        ):
            raise TypeError("direct_requirements must be a tuple of strings")

In [ ]:
def collect_environment_manifest(project_directory: Path) -> EnvironmentManifest:
    """Collect software provenance from the current Python process.

    Parameters
    ----------
    project_directory : Path
        Verified project root containing ``pyproject.toml`` and ``uv.lock``.

    Returns
    -------
    EnvironmentManifest
        Validated snapshot of the active software environment.

    Raises
    ------
    OSError
        If project metadata cannot be read.
    KeyError
        If required project metadata is absent.
    ModuleNotFoundError
        If the course import package is unavailable.
    """

    with (project_directory / "pyproject.toml").open(mode="rb") as handle:
        document = tomllib.load(handle)
    distribution_value = document["project"]["name"]
    if not isinstance(distribution_value, str):
        raise TypeError("project.name must be a string")

    module_name = distribution_value.replace("-", "_")
    module = importlib.import_module(module_name)
    module_file = getattr(module, "__file__", None)
    if not isinstance(module_file, str):
        raise ValueError(f"{module_name!r} has no file-backed import origin")

    requirements = document["project"].get("dependencies", [])
    if not isinstance(requirements, list) or not all(
        isinstance(requirement, str) for requirement in requirements
    ):
        raise TypeError("project.dependencies must be a list of strings")

    return EnvironmentManifest(
        project_distribution=distribution_value,
        project_version=importlib.metadata.version(distribution_value),
        python_version=platform.python_version(),
        platform_description=platform.platform(),
        environment_prefix=str(Path(sys.prefix).resolve()),
        import_package=module_name,
        import_origin=str(Path(module_file).resolve()),
        lock_sha256=sha256_file(project_directory / "uv.lock"),
        direct_requirements=tuple(requirements),
    )


environment_manifest = collect_environment_manifest(project_root)
print(environment_manifest)

assert environment_manifest.project_distribution == "rice-dsm"
assert environment_manifest.import_package == "rice_dsm"
assert Path(environment_manifest.import_origin).is_relative_to(expected_source)

### Serialize derived evidence without overwriting project files

`dataclasses.asdict` produces JSON-compatible built-ins except that the tuple is encoded as a JSON
array. We write to a temporary directory because this lesson should not create untracked artifacts.
A real experiment would store the manifest beside its derived results under an explicit policy.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    manifest_path = Path(temporary_directory) / "environment-manifest.json"
    manifest_path.write_text(
        json.dumps(asdict(environment_manifest), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    restored_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(manifest_path.read_text(encoding="utf-8"))

assert restored_manifest["lock_sha256"] == environment_manifest.lock_sha256
assert restored_manifest["project_version"] == environment_manifest.project_version

### Interpret the manifest honestly

This record can help determine whether two runs used the same locked software and project source
location. It does **not** yet capture:

- Git commit and uncommitted changes;
- input-data identities and transformations;
- model configuration, random seeds, or trained parameters;
- CPU, GPU, numerical backend, or thread settings;
- external services and credentials; or
- whether the method and measurements answer the scientific question.

Provenance must match the causal risks of the computation.

## Debugging: inspect boundaries in order

Use this sequence instead of repeatedly installing packages:

1. **Location:** Is the terminal or kernel inside the intended repository?
2. **Declaration:** Does `pyproject.toml` contain the requirement and build system?
3. **Resolution:** Does `uv lock --check` accept the existing lockfile?
4. **Synchronization:** Does `uv sync --locked` reproduce `.venv`?
5. **Kernel:** Does `sys.prefix` equal this repository's `.venv`?
6. **Distribution:** Does `importlib.metadata.version(name)` find the installed project?
7. **Import:** Does `find_spec(name)` point to the intended source?
8. **Process state:** Could a restart clear stale imports or out-of-order variables?

Record the first broken boundary and the exact evidence shown there.

### Common failure patterns

| Symptom | Likely boundary | Useful evidence |
| --- | --- | --- |
| `uv` is not recognized | shell command search | `uv --version` |
| no project found | working directory | `pwd` / `Get-Location` and file listing |
| lockfile rejected | declaration vs resolution | `uv lock --check` output |
| terminal import works, notebook fails | selected kernel | `sys.prefix` in both processes |
| import resolves to another checkout | import origin | `module.__file__` |
| source edit appears ignored | process cache | restart kernel, then Run All |
| name exists only after manual cell order | notebook state | Restart and Run All |

On Windows, macOS, and Linux, use `uv run` and Python diagnostics rather than assuming activation or
path syntax.

## Guided practice: write one environment health check

Complete a function that returns messages rather than printing or raising at the first problem. This
lets a user see all independent checks in one report.

Success criteria: detect Python version, virtual-environment status, course `.venv`, project import,
and import origin; return an immutable tuple; include no personal hard-coded path.

In [ ]:
def environment_health(project_directory: Path) -> tuple[str, ...]:
    """Return diagnostic messages for the active course environment.

    Parameters
    ----------
    project_directory : Path
        Verified course repository root.

    Returns
    -------
    tuple of str
        Empty when all checks pass; otherwise one message per failed check.
    """

    problems: list[str] = []
    expected_prefix = (project_directory / ".venv").resolve()
    if sys.version_info[:2] != (3, 12):
        problems.append("Python must be version 3.12 for this course")
    if sys.prefix == sys.base_prefix:
        problems.append("Python is not running inside a virtual environment")
    if Path(sys.prefix).resolve() != expected_prefix:
        problems.append(f"active prefix is not course environment: {sys.prefix}")

    specification = importlib.util.find_spec("rice_dsm")
    if specification is None or specification.origin is None:
        problems.append("rice_dsm cannot be located")
    elif not Path(specification.origin).resolve().is_relative_to(
        (project_directory / "src" / "rice_dsm").resolve()
    ):
        message = f"rice_dsm resolves to unexpected source: {specification.origin}"
        problems.append(message)
    return tuple(problems)


health_problems = environment_health(project_root)
print(health_problems or "All environment checks passed.")
assert health_problems == ()

## Independent practice: audit selected versions

Write a function that accepts distribution names and returns three groups: matching, mismatching,
and absent. Reuse `observe_versions`; do not parse requirement strings yourself.

Success criteria: immutable return values, deterministic input order, an unknown distribution test,
and a docstring explaining that the active platform installs only a subset of a universal lockfile.

In [ ]:
def partition_version_observations(
    observations: Iterable[VersionObservation],
) -> dict[str, tuple[str, ...]]:
    """Partition version observations by agreement state.

    Parameters
    ----------
    observations : iterable of VersionObservation
        Comparisons for packages relevant to the active platform.

    Returns
    -------
    dict of str to tuple of str
        Distribution names grouped as ``matching``, ``mismatching``, or ``absent``.

    Notes
    -----
    A universal lockfile can include records not selected for this platform. Callers
    should request only distributions expected in the active environment.
    """

    groups: dict[str, list[str]] = {
        "matching": [],
        "mismatching": [],
        "absent": [],
    }
    for observation in observations:
        if observation.installed_version is None:
            group = "absent"
        elif observation.agrees:
            group = "matching"
        else:
            group = "mismatching"
        groups[group].append(observation.distribution_name)
    return {name: tuple(values) for name, values in groups.items()}


practice_observations = observe_versions(
    ("pytest", "definitely-not-a-real-distribution"),
    locked_versions,
)
version_groups = partition_version_observations(practice_observations)
print(version_groups)

assert version_groups["matching"] == ("pytest",)
assert version_groups["absent"] == ("definitely-not-a-real-distribution",)

## Extension: design provenance for a real ML experiment

Extend `EnvironmentManifest` for one future course project. Decide which evidence belongs inside the
manifest and which should be referenced by a durable identifier:

- Git commit plus dirty-worktree status;
- raw and processed data hashes;
- train/validation/test split identity;
- complete model and preprocessing configuration;
- random seeds and deterministic-algorithm settings;
- CPU/GPU model and numerical-library backend;
- trained-model artifact and evaluation report; and
- human decisions that cannot be reconstructed from code.

There is no universally complete manifest. Justify the scope from the experiment's plausible failure
modes, privacy constraints, storage cost, and required reproducibility level.

## Retrieval practice

Answer without running code:

1. Why are the repository, project, `.venv`, and kernel four different objects?
2. What does `pyproject.toml` declare that `uv.lock` resolves?
3. How do exact `uv sync` and default `uv run` treat extraneous packages differently?
4. Why can a universal lockfile contain a package absent from your machine?
5. What evidence distinguishes the installed distribution from the imported module?
6. Why is `sys.prefix` usually more informative than searching `sys.executable` for `.venv`?
7. What does editable installation change, and what can a running kernel still cache?
8. Which reproducibility risks are absent from our environment manifest?

## Takeaway

Reproducibility is a chain of explicit states:

```text
declared requirements → locked resolution → synchronized environment
    → selected kernel → imported source → recorded provenance
```

When the chain breaks, diagnose the earliest failed boundary. Do not guess, modify `sys.path`, or
install into an unknown interpreter. The everyday student command remains:

```text
uv run python scripts/setup_course.py
```

Next, Notebook 01 turns scripts into modules and packages with deliberate public interfaces.

## Further reading

- [uv: Working on projects](https://docs.astral.sh/uv/guides/projects/)
- [uv: Locking and syncing](https://docs.astral.sh/uv/concepts/projects/sync/)
- [Python Packaging User Guide: Writing `pyproject.toml`](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/)
- [Python Packaging User Guide: Distribution versus import package](https://packaging.python.org/en/latest/discussions/distribution-package-vs-import-package/)
- [Python: `importlib.metadata`](https://docs.python.org/3/library/importlib.metadata.html)
- [Python: `sys.prefix` and `sys.base_prefix`](https://docs.python.org/3/library/sys.html#sys.prefix)
- [VS Code: Manage Jupyter kernels](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)
- [Course guide: How the Python project works](../../supplementary-materials/computing-foundations/06-how-the-course-python-project-works.md)